In [5]:
import sys
import pandas as pd
import numpy as np
import os
import re
import glob
from multiprocessing import Pool

PATH = "/home/jmurga/mkt/202004"
sys.path.insert(0, PATH + '/scripts/src/')
from pyAmkt import *
from slimParser import inputSlim, runSlim, parsePolDiv

# No demography simulations

## N = 1000

### Solving values to simulate with Analytical.jl

In [3]:
!which julia

/home/jmurga/julia-1.0.5/bin/julia


In [27]:
!julia /home/jmurga/mkt/202004/scripts/src/simTable.jl 1000

In [6]:
simTable = pd.read_csv(PATH + "/rawData/simulations/simTable.tsv", sep='\t')
tmpSim = simTable[(simTable.alpha==0.4) & (simTable.B!=0.4) & (simTable.B!=0.8)];
tmpSim = tmpSim.tail(3).head(1)
tmpSim['path'] = tmpSim.apply(lambda row: "/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_" + str(row['alpha']) + "_" + str(row['alphaW']) + "_" + str(np.round(row['B'],4)),axis=1)

### Running SLiM

Generating replicates and folders

In [8]:
bgs = []
for index,row in tmpSim.iterrows():
    os.makedirs(row.path,exist_ok=True)    
    os.makedirs(row.path + '/daf',exist_ok=True)    
    os.makedirs(row.path + '/div',exist_ok=True)    

    tmp = inputSlim((1,10000),"/home/jmurga/mkt/202004/scripts/slimRecipes/bgs.slim",10,500,1000, row.pposL,row.pposH,row.bgsThetaF,row.path,"/home/jmurga/.conda/envs/abc-mk/bin/slim")
    bgs.append(tmp)

In [ ]:
p = Pool(processes=20)
output = p.starmap(runSlim,bgs[0])
p.terminate()
output1 = p.starmap(runSlim,bgs[1])
output2 = p.starmap(runSlim,bgs[2])
output3 = p.starmap(runSlim,bgs[3])
output4 = p.starmap(runSlim,bgs[4])
output5 = p.starmap(runSlim,bgs[5])
p.terminate()

### Processing simulated data

In [ ]:
al = []
for index,row in tmpSim.iterrows():
        
    try:
        sfs, div ,alphas = parsePolDiv(row.path,661)
        sfs.to_csv(row.path + "/sfs.tsv",header=True,index=False,sep="\t")
        div.to_csv(row.path + "/div.tsv",header=True,index=False,sep="\t")
        alphas.to_csv(row.path + "/alphas.tsv",header=True,index=False,sep="\t")
                
        try:
            cSfs = cumulativeSfs(sfs.to_numpy())
            asymp = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)
        except:
            cSfs = cumulativeSfs(reduceSfs(sfs.to_numpy(),100))
            asymp = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)

        df = pd.DataFrame(np.mean(alphas,axis=0)).T
        df['asymp'] = asymp['alpha']
        df['analyticalEstimation']= row.estimation
        al.append(df)
    except:
        continue
df = pd.concat(al)
df.set_index(tmpSim.path.apply(lambda row: row.split('/')[-1]))

### Analysis

In [ ]:
priorsJulia(tmpSim,3000,20,"/home/jmurga/mkt/202004/scripts/src/sim.jl",threads=20)

In [ ]:
%%bash 
PATH="/home/jmurga/mkt/202004"
parallel -j10 < ${PATH}/rawData/summStat/noDemog/noDemog_0.4_0.2_0.999/job.txt

In [ ]:
for index,row in tmpSim.iterrows():
    summStatPath = re.sub('simulations','summStat',row.path)
    outputs = glob.glob(summStatPath + '/slim*')
    
    l=[]
    for f in outputs:
        n = np.loadtxt(f)
        nm = np.mean(n,axis=0)
        l.append(nm)
    
    df = pd.DataFrame(l,columns=['trueAlphaW','trueAlphaS','trueAlpha'])
    df['analysis'] = 'abc'
    alphas = pd.read_csv(row.path + '/alphas.tsv',sep='\t')
    alphas['analysis'] = 'simulations'
    
    result = 

## N = 100. Rescaling recipe

### Solving values to simulate with Analytical.jl

In [3]:
!which julia

/home/jmurga/julia-1.0.5/bin/julia


In [27]:
!julia /home/jmurga/mkt/202004/scripts/src/simTable.jl 100 simTableRescaled

In [15]:
simTable = pd.read_csv(PATH + "/rawData/simulations/simTableRescaled.tsv", sep='\t')
tmpSim = simTable[(simTable.alpha==0.4) & (simTable.B!=0.4) & (simTable.B!=0.8)];
tmpSim

,bgsThetaF,pposL,pposH,alphaW,alpha,estimation,B
24,6.608754e-06,0.004054,0.001198,0.1,0.4,0.25688,0.200
25,6.608754e-06,0.008093,0.000797,0.2,0.4,0.23088,0.200
26,6.608754e-06,0.012118,0.000398,0.3,0.4,0.20289,0.200
27,3.762519e-06,0.004054,0.001198,0.1,0.4,0.31341,0.400
28,3.762519e-06,0.008093,0.000797,0.2,0.4,0.29570,0.400
29,3.762519e-06,0.012118,0.000398,0.3,0.4,0.27699,0.400
33,4.108000e-09,0.004054,0.001198,0.1,0.4,0.40182,0.999
34,4.108000e-09,0.008093,0.000797,0.2,0.4,0.40292,0.999
35,4.108000e-09,0.012118,0.000398,0.3,0.4,0.40402,0.999


In [11]:
tmpSim['path'] = tmpSim.apply(lambda row: "/home/jmurga/mkt/202004/rawData/simulations/noDemogRescaled/noDemogRescaled_" + str(row['alpha']) + "_" + str(row['alphaW']) + "_" + str(np.round(row['B'],4)),axis=1)

/home/jmurga/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


Generating replicates and folders

In [ ]:
bgs = []
for index,row in tmpSim.iterrows():
    os.makedirs(row.path,exist_ok=True)    
    os.makedirs(row.path + '/daf',exist_ok=True)    
    os.makedirs(row.path + '/div',exist_ok=True)    

    tmp = inputSlim((0,100000),"/home/jmurga/mkt/202004/scripts/slimRecipes/bgsRescaled.slim",10,500,100, row.pposL,row.pposH,row.bgsThetaF,row.path,"/home/jmurga/.conda/envs/abc-mk/bin/slim")
    bgs.append(tmp)

In [ ]:
bgs[0][-1]

In [ ]:
p = Pool(processes=20)
%time output = p.starmap(runSlim,bgs[0])
output1 = p.starmap(runSlim,bgs[1])
p.terminate()
output2 = p.starmap(runSlim,bgs[2])
output3 = p.starmap(runSlim,bgs[3])
output4 = p.starmap(runSlim,bgs[4])
output5 = p.starmap(runSlim,bgs[5])
p.terminate()

Processing simulated data

In [8]:
al = []
for index,row in tmpSim.iterrows():
        
    try:
        sfs, div ,alphas = parsePolDiv(row.path,661)
        sfs.to_csv(row.path + "/sfs.tsv",header=True,index=False,sep="\t")
        div.to_csv(row.path + "/div.tsv",header=True,index=False,sep="\t")
        alphas.to_csv(row.path + "/alphas.tsv",header=True,index=False,sep="\t")
        
        al.append(pd.DataFrame(np.mean(alphas,axis=0).append(pd.Series(row.estimation,index=["analyticalEstimation"]))).T)
    except:
        continue
df = pd.concat(al)
df.set_index(tmpSim.path.apply(lambda row: row.split('/')[-1]))

100%|██████████| 1001/1001 [00:08<00:00, 115.47it/s]


,trueAlphaW,trueAlphaS,trueAlpha,analyticalEstimation
path,,,,
noDemog_0.4_0.1_0.2,0.035355,0.164530,0.199885,0.25700
noDemog_0.4_0.2_0.2,0.075022,0.111632,0.186654,0.23098
noDemog_0.4_0.3_0.2,0.112618,0.057236,0.169854,0.20298
noDemog_0.4_0.1_0.999,0.085568,0.306443,0.392011,0.40150
noDemog_0.4_0.2_0.999,0.173646,0.206755,0.380402,0.40263
noDemog_0.4_0.3_0.999,0.264382,0.105483,0.369864,0.40377


In [9]:
# noDemog_0.4_0.2_0.999
sfs = pd.read_csv("/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_0.4_0.2_0.999/sfs.tsv",header=0,sep="\t");sfs

,f,pi,p0,pw,pi_nopos
0,0.0008,4898.0,2054.0,43.0,4855.0
1,0.0015,1951.0,930.0,16.0,1935.0
2,0.0023,1224.0,641.0,15.0,1209.0
3,0.0030,853.0,479.0,9.0,844.0
4,0.0038,667.0,412.0,12.0,655.0
...,...,...,...,...,...
1316,0.9962,2.0,2.0,1.0,1.0
1317,0.9970,4.0,0.0,1.0,3.0
1318,0.9977,0.0,1.0,0.0,0.0
1319,0.9985,2.0,3.0,0.0,2.0


In [ ]:
cSfs = cumulativeSfs(sfs.to_numpy())

In [ ]:
amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)

### Running SLiM

Generating replicates and folders

In [4]:
import os
from multiprocessing import Pool

In [ ]:
bgs = []
for index,row in tmpSim.iterrows():
    os.makedirs(row.path,exist_ok=True)    
    os.makedirs(row.path + '/daf',exist_ok=True)    
    os.makedirs(row.path + '/div',exist_ok=True)    

    tmp = inputSlim((0,1000),"/home/jmurga/mkt/202004/scripts/slimRecipes/bgsVar.slim",10,500,1000, row.pposL,row.pposH,row.bgsThetaF,row.path)
    bgs.append(tmp)

In [ ]:
bgs[0][-1]

In [ ]:
p = Pool(processes=20)
output = p.starmap(runSlim,bgs[0])
output1 = p.starmap(runSlim,bgs[1])
output2 = p.starmap(runSlim,bgs[2])
output3 = p.starmap(runSlim,bgs[3])
output4 = p.starmap(runSlim,bgs[4])
output5 = p.starmap(runSlim,bgs[5])
p.terminate()

In [8]:
al = []
for index,row in tmpSim.iterrows():
        
    try:
        sfs, div ,alphas = parsePolDiv(row.path)
        sfs.to_csv(row.path + "/sfs.tsv",header=True,index=False,sep="\t")
        div.to_csv(row.path + "/div.tsv",header=True,index=False,sep="\t")
        alphas.to_csv(row.path + "/alphas.tsv",header=True,index=False,sep="\t")
        cSfs = cumulativeSfs(sfs.to_numpy())
        asymp = amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)
        
        al.append(pd.DataFrame(np.mean(alphas,axis=0).append(pd.Series(asymp['alpha'],index=["asymp"]),pd.Series(row.estimation,index=["analyticalEstimation"]))).T)
    except:
        continue
df = pd.concat(al)
df.set_index(tmpSim.path.apply(lambda row: row.split('/')[-1]))

100%|██████████| 1001/1001 [00:08<00:00, 115.47it/s]


,trueAlphaW,trueAlphaS,trueAlpha,analyticalEstimation
path,,,,
noDemog_0.4_0.1_0.2,0.035355,0.164530,0.199885,0.25700
noDemog_0.4_0.2_0.2,0.075022,0.111632,0.186654,0.23098
noDemog_0.4_0.3_0.2,0.112618,0.057236,0.169854,0.20298
noDemog_0.4_0.1_0.999,0.085568,0.306443,0.392011,0.40150
noDemog_0.4_0.2_0.999,0.173646,0.206755,0.380402,0.40263
noDemog_0.4_0.3_0.999,0.264382,0.105483,0.369864,0.40377


In [9]:
# noDemog_0.4_0.2_0.999
sfs = pd.read_csv("/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_0.4_0.2_0.999/sfs.tsv",header=0,sep="\t");sfs

,f,pi,p0,pw,pi_nopos
0,0.0008,4898.0,2054.0,43.0,4855.0
1,0.0015,1951.0,930.0,16.0,1935.0
2,0.0023,1224.0,641.0,15.0,1209.0
3,0.0030,853.0,479.0,9.0,844.0
4,0.0038,667.0,412.0,12.0,655.0
...,...,...,...,...,...
1316,0.9962,2.0,2.0,1.0,1.0
1317,0.9970,4.0,0.0,1.0,3.0
1318,0.9977,0.0,1.0,0.0,0.0
1319,0.9985,2.0,3.0,0.0,2.0


In [ ]:
amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)

### Solving values to simulate with Analytical.jl

In [3]:
!which julia

/home/jmurga/julia-1.0.5/bin/julia


In [27]:
!julia /home/jmurga/mkt/202004/scripts/src/simTable.jl 1000

In [4]:
simTable = pd.read_csv(PATH + "/rawData/simulations/simTable.tsv", sep='\t')
tmpSim = simTable[(simTable.alpha==0.4) & (simTable.B!=0.4) & (simTable.B!=0.8)];
tmpSim

,bgsThetaF,pposL,pposH,alphaW,alpha,estimation,B
24,6.608750e-07,0.003877,0.000311,0.1,0.4,0.25700,0.200
25,6.608750e-07,0.007739,0.000207,0.2,0.4,0.23098,0.200
26,6.608750e-07,0.011586,0.000103,0.3,0.4,0.20298,0.200
33,4.110000e-10,0.003877,0.000311,0.1,0.4,0.40150,0.999
34,4.110000e-10,0.007739,0.000207,0.2,0.4,0.40263,0.999
35,4.110000e-10,0.011586,0.000103,0.3,0.4,0.40377,0.999


In [7]:
tmpSim['path'] = tmpSim.apply(lambda row: "/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_" + str(row['alpha']) + "_" + str(row['alphaW']) + "_" + str(np.round(row['B'],4)),axis=1)

/home/jmurga/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.


### Running SLiM

Generating replicates and folders

In [4]:
import os
from multiprocessing import Pool

In [ ]:
bgs = []
for index,row in tmpSim.iterrows():
    os.makedirs(row.path,exist_ok=True)    
    os.makedirs(row.path + '/daf',exist_ok=True)    
    os.makedirs(row.path + '/div',exist_ok=True)    

    tmp = inputSlim((0,1000),"/home/jmurga/mkt/202004/scripts/slimRecipes/bgsVar.slim",10,500,1000, row.pposL,row.pposH,row.bgsThetaF,row.path)
    bgs.append(tmp)

In [ ]:
bgs[0][-1]

In [ ]:
p = Pool(processes=20)
output = p.starmap(runSlim,bgs[0])
output1 = p.starmap(runSlim,bgs[1])
output2 = p.starmap(runSlim,bgs[2])
output3 = p.starmap(runSlim,bgs[3])
output4 = p.starmap(runSlim,bgs[4])
output5 = p.starmap(runSlim,bgs[5])
p.terminate()

Processing simulated data

In [8]:
al = []
for index,row in tmpSim.iterrows():
        
    try:
        sfs, div ,alphas = parsePolDiv(row.path)
        sfs.to_csv(row.path + "/sfs.tsv",header=True,index=False,sep="\t")
        div.to_csv(row.path + "/div.tsv",header=True,index=False,sep="\t")
        alphas.to_csv(row.path + "/alphas.tsv",header=True,index=False,sep="\t")
        
        al.append(pd.DataFrame(np.mean(alphas,axis=0).append(pd.Series(row.estimation,index=["analyticalEstimation"]))).T)
    except:
        continue
df = pd.concat(al)
df.set_index(tmpSim.path.apply(lambda row: row.split('/')[-1]))

100%|██████████| 1001/1001 [00:08<00:00, 115.47it/s]


,trueAlphaW,trueAlphaS,trueAlpha,analyticalEstimation
path,,,,
noDemog_0.4_0.1_0.2,0.035355,0.164530,0.199885,0.25700
noDemog_0.4_0.2_0.2,0.075022,0.111632,0.186654,0.23098
noDemog_0.4_0.3_0.2,0.112618,0.057236,0.169854,0.20298
noDemog_0.4_0.1_0.999,0.085568,0.306443,0.392011,0.40150
noDemog_0.4_0.2_0.999,0.173646,0.206755,0.380402,0.40263
noDemog_0.4_0.3_0.999,0.264382,0.105483,0.369864,0.40377


In [9]:
# noDemog_0.4_0.2_0.999
sfs = pd.read_csv("/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_0.4_0.2_0.999/sfs.tsv",header=0,sep="\t");sfs

,f,pi,p0,pw,pi_nopos
0,0.0008,4898.0,2054.0,43.0,4855.0
1,0.0015,1951.0,930.0,16.0,1935.0
2,0.0023,1224.0,641.0,15.0,1209.0
3,0.0030,853.0,479.0,9.0,844.0
4,0.0038,667.0,412.0,12.0,655.0
...,...,...,...,...,...
1316,0.9962,2.0,2.0,1.0,1.0
1317,0.9970,4.0,0.0,1.0,3.0
1318,0.9977,0.0,1.0,0.0,0.0
1319,0.9985,2.0,3.0,0.0,2.0


In [ ]:
cSfs = cumulativeSfs(sfs.to_numpy())

In [ ]:
amkt(cSfs[:,[0,1,2]],div.to_numpy().flatten(),0,1)
amkt(cSfs[:,[0,4,2]],div.to_numpy().flatten(),0,1)

# Tennesen model

## Solving values to simulate with Analytical.jl

In [3]:
!which julia

/home/jmurga/julia-1.0.5/bin/julia


In [27]:
!julia /home/jmurga/mkt/202004/scripts/src/simTable.jl 7310 simTableTennesen

In [9]:
simTable = pd.read_csv(PATH + "/rawData/simulations/simTableTennesen.tsv", sep='\t')
tmpSim = simTable[(simTable.alpha==0.4) & (simTable.alphaW==0.2) & (simTable.B!=0.4) & (simTable.B!=0.8) & (simTable.B!=0.2)];
tmpSim

,bgsThetaF,pposL,pposH,alphaW,alpha,estimation,B
24,9.040700e-08,0.003860,0.000239,0.1,0.4,0.25701,0.200
25,9.040700e-08,0.007705,0.000159,0.2,0.4,0.23100,0.200
26,9.040700e-08,0.011536,0.000079,0.3,0.4,0.20299,0.200
33,5.600000e-11,0.003860,0.000239,0.1,0.4,0.40147,0.999
34,5.600000e-11,0.007705,0.000159,0.2,0.4,0.40261,0.999
35,5.600000e-11,0.011536,0.000079,0.3,0.4,0.40374,0.999


In [38]:
import os
from multiprocessing import Pool

In [ ]:
tmpSim['path'] = tmpSim.apply(lambda row: "/home/jmurga/mkt/202004/rawData/simulations/tennesen/tennesen_" + str(row['alpha']) + "_" + str(row['alphaW']) + "_" + str(np.round(row['B'],4)),axis=1)

In [101]:
bgs = []
for index,row in tmpSim.iterrows():
    os.makedirs(row.path,exist_ok=True)    
    os.makedirs(row.path + '/daf',exist_ok=True)    
    os.makedirs(row.path + '/div',exist_ok=True)    

    tmp = inputSlim((0,100),"/home/jmurga/mkt/202004/scripts/slimRecipes/tennesenBgs.slim",10,500,7310, row.pposL,row.pposH,row.bgsThetaF,row.path,"/home/jmurga/anaconda3/envs/abc-mk/bin/slim")
    bgs.append(tmp)

In [102]:
bgs[0][-1]

('/home/jmurga/mkt/202004/scripts/slimRecipes/bgsVar.slim',
 10,
 500,
 1000,
 0.003876632857,
 0.000311470804,
 6.60875e-07,
 '/home/jmurga/mkt/202004/rawData/simulations/noDemog/noDemog_0.4_0.1_0.2',
 10000)

In [ ]:
p = Pool(processes=5)
output = p.starmap(runSlim,bgs[0])
p.terminate()